In [1]:
%%capture
!pip install -q transformers==4.40.1 sktime


In [2]:
import gc
import json
import math
import os
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import transformers
from transformers import AutoModelForCausalLM
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from sktime.performance_metrics.forecasting import MeanAbsoluteScaledError

EXPECTED_TRANSFORMERS_VERSION = "4.40.1"
print("Transformers version:", transformers.__version__)
print("PyTorch version:", torch.__version__)
assert transformers.__version__ == EXPECTED_TRANSFORMERS_VERSION, (
    f"Expected transformers {EXPECTED_TRANSFORMERS_VERSION}, "
    f"but found {transformers.__version__}. Restart the runtime after installation."
)


Transformers version: 4.40.1
PyTorch version: 2.6.0+cu124


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Configuration


Set paths and benchmark configuration.


In [4]:
CONFIG_PATH = '/content/drive/MyDrive/Models parameters JSON files/seasonalnaive.json'
OUTPUT_DIR = '/content/drive/MyDrive/results/json/Timer'
MODEL_ID = 'thuml/timer-base-84m'
TRAIN_RATIO = 0.80
SHOW_PLOTS = True

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    CONFIG = json.load(f)

DATASETS = CONFIG.get('datasets', [])
TOTAL_TARGETS = sum(len(dataset.get('targets', [])) for dataset in DATASETS)
total_models = TOTAL_TARGETS
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)
model = model.to(device)
model.eval()

print('Model loaded successfully.')
print('Input token length:', model.config.input_token_len)
print('Output token lengths:', model.config.output_token_lens)


Device: cpu


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

configuration_timer.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/thuml/timer-base-84m:
- configuration_timer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

ts_generation_mixin.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/thuml/timer-base-84m:
- ts_generation_mixin.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/thuml/timer-base-84m:
- modeling_timer.py
- ts_generation_mixin.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/337M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Model loaded successfully.
Input token length: 96
Output token lengths: [96]


## Split data into training and test parts


In [6]:
# Cell 2: Split data into training and test parts
def split_data(dates, values):
    split_index = int(len(values) * TRAIN_RATIO)

    train_values = values[:split_index]
    test_values = values[split_index:]
    train_dates = dates.iloc[:split_index]
    test_dates = dates.iloc[split_index:]

    return train_dates, train_values, test_dates, test_values

## Metrics and JSON output

In [7]:
def scale_values(y_train, y_true, y_pred):
    y_train = np.asarray(y_train, dtype=np.float64).reshape(-1, 1)
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1, 1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1, 1)

    scaler = MinMaxScaler()
    scaler.fit(y_train)
    y_true_scaled = scaler.transform(y_true)
    y_pred_scaled = scaler.transform(y_pred)

    return y_true_scaled, y_pred_scaled


In [8]:
# Cell 4: Calculate forecast errors
def calculate_metrics(train_values, test_values, predictions):
    test_array = np.asarray(test_values, dtype=np.float64)
    prediction_array = np.asarray(predictions, dtype=np.float64)

    mase = MeanAbsoluteScaledError()(
        test_array, prediction_array, y_train=train_values
    )

    # Unscaled errors use the original values.
    mae = float(mean_absolute_error(test_array, prediction_array))
    mse = float(mean_squared_error(test_array, prediction_array))
    rmse = float(np.sqrt(mse))

    # Scaled errors use a MinMaxScaler fitted only on the training values.
    test_scaled, predictions_scaled = scale_values(
        train_values, test_array, prediction_array
    )
    scaled_mae = float(mean_absolute_error(test_scaled, predictions_scaled))
    scaled_mse = float(mean_squared_error(test_scaled, predictions_scaled))
    scaled_rmse = float(np.sqrt(scaled_mse))

    return {
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'mase': float(mase),
        'mae_scaled': scaled_mae,
        'mse_scaled': scaled_mse,
        'rmse_scaled': scaled_rmse,
    }


## Time-series JSON output

In [9]:
def save_timeseries_full(
    log_path,
    model_name,
    full_dates,
    full_values,
    train_dates,
    train_values,
    test_dates,
    test_values,
    forecast_dates,
    forecast_values,
    metrics
):
    # Read previous results from the JSON file.
    if os.path.exists(log_path):
        with open(log_path, 'r', encoding='utf-8') as file:
            log_data = json.load(file)
    else:
        log_data = []

    # Create the result for this model.
    entry = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model': model_name,
        'full_series': {
            'dates': [str(date) for date in full_dates],
            'values': [float(value) for value in full_values]
        },
        'train_series': {
            'dates': [str(date) for date in train_dates],
            'values': [float(value) for value in train_values]
        },
        'test_series': {
            'dates': [str(date) for date in test_dates],
            'values': [float(value) for value in test_values]
        },
        'forecast': {
            'dates': [str(date) for date in forecast_dates],
            'prediction': [float(value) for value in forecast_values]
        },
        'metrics': {
            'mae': float(metrics['mae']),
            'mse': float(metrics['mse']),
            'rmse': float(metrics['rmse']),
            'mae_scaled': float(metrics['mae_scaled']),
            'mse_scaled': float(metrics['mse_scaled']),
            'rmse_scaled': float(metrics['rmse_scaled']),
            'mase': None if np.isnan(metrics['mase']) else float(metrics['mase'])
        }
    }

    # Add the new result and save it.
    log_data.append(entry)

    with open(log_path, 'w', encoding='utf-8') as file:
        json.dump(log_data, file, indent=4, ensure_ascii=False)


## Prepare one target column

In [10]:
# Cell 1: Prepare one target column
def prepare_target(data, dataset, target):
    dataset_name = dataset['dataset_name']
    time_column = dataset['time_column']
    target_column = target['column']

    target_data = data[[time_column, target_column]].copy()
    target_data[target_column] = pd.to_numeric(target_data[target_column])

    dates = target_data[time_column]
    values = target_data[target_column].to_numpy(dtype=float)

    return dataset_name, time_column, target_column, dates, values

## Create Timer forecast

In [11]:
def create_forecast(train_values, test_values):
    prediction_length = len(test_values)

    if prediction_length == 0:
        return np.array([], dtype=np.float32)

    seqs = torch.as_tensor(
        train_values,
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    # Timer generates forecasts in fixed-size output blocks (96 for timer-base-84m).
    token_len = int(model.config.output_token_lens[0])
    generation_length = math.ceil(prediction_length / token_len) * token_len

    with torch.no_grad():
        output = model.generate(
            seqs,
            max_new_tokens=generation_length
        )

    output = output.squeeze(0).detach().cpu().numpy()
    predictions = output[:prediction_length]

    if len(predictions) != prediction_length:
        raise RuntimeError(
            f'Prediction length ({len(predictions)}) does not match '
            f'test length ({prediction_length}).'
        )

    del seqs, output
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return predictions


## Forecast, evaluate, and save each target

In [12]:
# Cell 7: Run all steps for one target column
def run_target(data, dataset, target, model_number):
    dataset_name, time_column, target_column, dates, values = prepare_target(
        data, dataset, target
    )

    train_dates, train_values, test_dates, test_values = split_data(dates, values)
    predictions = create_forecast(train_values, test_values)
    metrics = calculate_metrics(train_values, test_values, predictions)

    print(f'\nModel {model_number}/{total_models}: {dataset_name} - {target_column}')
    print('MAE scaled:', round(metrics['mae_scaled'], 4))
    print('RMSE scaled:', round(metrics['rmse_scaled'], 4))
    print('MASE:', round(metrics['mase'], 4))

    show_forecast(
        train_dates, train_values, test_dates, test_values, predictions,
        dataset_name, time_column, target_column
    )

    save_result(
        dataset_name, target_column, target, dates, values,
        train_dates, train_values, test_dates, test_values,
        predictions, metrics
    )


## Plot target forecast

In [13]:
# Cell 5: Draw the forecast chart
def show_forecast(train_dates, train_values, test_dates, test_values, predictions,
                  dataset_name, time_column, target_column):
    if SHOW_PLOTS:
        plt.figure(figsize=(16, 6))
        plt.plot(train_dates, train_values, label='Training data')
        plt.plot(test_dates, test_values, label='Actual values')
        plt.plot(test_dates, predictions, '--', label='Timer forecast')
        plt.title(f'{dataset_name} - {target_column}')
        plt.xlabel(time_column)
        plt.ylabel(target_column)
        plt.legend()
        plt.grid()
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


## Save target output

In [14]:
def save_result(dataset_name, target_column, target, dates, values,
                train_dates, train_values, test_dates, test_values,
                predictions, metrics):
    output_file = target['output_file']
    log_path = os.path.join(OUTPUT_DIR, output_file)
    model_name = f'Timer_{dataset_name}_{target_column}'

    save_timeseries_full(
        log_path, model_name, dates, values,
        train_dates, train_values, test_dates, test_values,
        test_dates, predictions, metrics
    )

    all_results.append({
        'dataset': dataset_name,
        'target': target_column,
        'model': model_name,
        **metrics
    })

    print('JSON saved:', log_path)


## Run all datasets and targets

In [15]:


# Cell 8: Run all datasets and target columns
all_results = []
completed_models = 0

for dataset in DATASETS:
    dataset_name = dataset['dataset_name']
    file_path = dataset['file_path']
    time_column = dataset['time_column']

    data = pd.read_csv(file_path)
    data[time_column] = pd.to_datetime(data[time_column])
    data = data.sort_values(time_column).reset_index(drop=True)

    print('\n' + '=' * 60)
    print('Dataset:', dataset_name)

    for target in dataset['targets']:
        completed_models += 1
        run_target(data, dataset, target, completed_models)

Output hidden; open in https://colab.research.google.com to view.

In [16]:
print('\n\n')
print('=' * 110)
print('ALL DATASETS / TARGETS COMPLETED')
print('=' * 110)
if len(all_results) > 0:
    summary_df = pd.DataFrame(all_results)
    print(summary_df.to_string(index=False))
else:
    print('No target was completed.')

print('\n')
print('=' * 90)
print('PROCESS FINISHED')
print('=' * 90)
print('Datasets requested:', len(DATASETS))
print('Targets requested:', total_models)
print('Targets completed:', len(all_results))
print('Output directory:', OUTPUT_DIR)
if len(all_results) != total_models:
    print('\nWARNING:')
    print('Not all requested targets completed successfully.')
    print('Requested:', total_models)
    print('Completed:', len(all_results))
else:
    print('ALL REQUESTED TARGETS COMPLETED.')





ALL DATASETS / TARGETS COMPLETED
dataset         target                      model           mae          mse          rmse      mase  mae_scaled  mse_scaled  rmse_scaled
   WB90             wt              Timer_WB90_wt      0.111931 2.948152e-02      0.171702  0.832835    0.041151    0.003985     0.063126
   WB90            toc             Timer_WB90_toc      0.040793 2.561452e-03      0.050611  0.476669    0.086793    0.011596     0.107683
   WB90    temperature     Timer_WB90_temperature      1.171836 1.680341e+00      1.296280  2.070192    0.123481    0.018658     0.136594
   WB90             ph              Timer_WB90_ph      0.060661 7.445033e-03      0.086285  0.847407    0.189565    0.072705     0.269639
   WB90       humidity        Timer_WB90_humidity      4.038421 3.143887e+01      5.607038  1.708252    0.085924    0.014232     0.119299
   WB90             do              Timer_WB90_do      0.072410 1.272000e-02      0.112783  1.038640    0.114937    0.032048     0.17902